In [2]:
import pymysql
from configparser import ConfigParser

config = ConfigParser()
config.read('../Chapter1/config.ini') # 指定設定檔的檔案路徑

connection = pymysql.connect(
    host=config.get('DB', 'host'),
    user=config.get('DB', 'user'),
    password=config.get('DB', 'password'),
    port=config.getint('DB', 'port'),
    cursorclass=pymysql.cursors.DictCursor,
)

print(connection.open)

True


建立資料庫

In [ ]:
database = "chapter2"
with connection.cursor() as cursor:
    sql = f"""
        CREATE DATABASE IF NOT EXISTS {database}              
    """
    # 沒有的話在建立 
    # 執行建立的 SQL 語句
    cursor.execute(sql)
    # 執行查看資料庫
    cursor.execute('SHOW DATABASES;')
    dbs = cursor.fetchall()
print(dbs)


[{'Database': 'chapter2'}, {'Database': 'classicmodels'}, {'Database': 'information_schema'}, {'Database': 'my_database'}, {'Database': 'my_titanic'}, {'Database': 'my_train_titanic'}, {'Database': 'mysql'}, {'Database': 'performance_schema'}, {'Database': 'sakila'}, {'Database': 'social_media_app'}, {'Database': 'sys'}, {'Database': 'transaction_test'}, {'Database': 'world'}]


建立資料表

In [ ]:
import pymysql
connection = pymysql.connect(
    host=config.get('DB', 'host'),
    user=config.get('DB', 'user'),
    password=config.get('DB', 'password'),
    port=config.getint('DB', 'port'),
    cursorclass=pymysql.cursors.DictCursor,
    database=database,
)

""" 建立 user 資料表
    id 主鍵
    name 字串 不能為空
    age 整數 
    username 字串 不能為空 必須唯一
    password 字串 不能為空
"""
with connection.cursor() as cursor:
    sql = """
        CREATE TABLE IF NOT EXISTS user (
            id INT AUTO_INCREMENT PRIMARY KEY,
            name VARCHAR(255) NOT NULL,
            age INT,
            username VARCHAR(200) NOT NULL UNIQUE,
            password VARCHAR(200) NOT NULL
        );
    """
    cursor.execute(sql)
    cursor.execute('SHOW TABLES')
    tables = cursor.fetchall()

print(tables)


[{'Tables_in_chapter2': 'user'}]


寫入資料

In [ ]:
from pprint import pprint
with connection.cursor() as cursor:
    sql = """
        INSERT INTO user (name, age, username, password)
        VALUES('Umi', 25, 'umi123', '123456')
    """
    # 執行寫入的 SQL 語句
    cursor.execute(sql)
    
    # 執行查詢資料表
    cursor.execute("SELECT * FROM user;")
    # 取得查詢的所有資料
    r = cursor.fetchall()

print(r)

OperationalError: (1054, "Unknown column 'password' in 'field list'")

In [ ]:
#如果沒有執行，資料只會存在記憶體，關閉程式或連線後就會消失，不會寫進資料庫
#INSERT UPDATE DELETE 會改變內容的操作都需要執行COMMIT
connection.commit()

使用另一個連線查詢資料庫

In [ ]:
connection2 = pymysql.connect(
    host=config.get('DB', 'host'),
    user=config.get('DB', 'user'),
    password=config.get('DB', 'password'),
    port=config.getint('DB', 'port'),
    cursorclass=pymysql.cursors.DictCursor,
    database=database,
)
with connection2.cursor() as cursor:
    cursor.execute("SELECT * FROM user;")
    result = cursor.fetchall()

pprint(result)
#如果上方沒寫入COMMIT 結果會是空的
connection2.close()

提交資料庫變更 `conncection.commit()`

In [ ]:
from pprint import pprint
with connection.cursor() as cursor:
    sql =sql = """
        INSERT INTO user (name, age, username, password)
        VALUES('Judy', 26, 'J456', '654321')
    """
    cursor.execute(sql)
    
    # 提交資料庫的變更
    row = cursor.rowcount #可檢查是否異動
    print(f'資料庫異動: {row} 行')
    if row == 1: #有異動才提交
        connection.commit()   

In [ ]:
from pprint import pprint
from pymysql.err import IntegrityError

with connection.cursor() as cursor:
    sql =sql = """
        INSERT INTO user (name, age, username, password)
        VALUES('Judy', 26, 'J456', '654321')
    """
    try: 
        cursor.execute(sql)
        
        # 提交資料庫的變更
        row = cursor.rowcount #可檢查是否異動
        print(f'資料庫異動: {row} 行')
        if row == 1: #有異動才提交
            connection.commit()   
    except IntegrityError:
        print('該 user 已建立')

更新資料

In [ ]:
with connection.cursor() as cursor:
    sql = """
        UPDATE user SET age = 35
        WHERE username = 'Umi';
    """
    cursor.execute(sql)
    
    # 可以搭配 rowcount 屬性來檢查受影響的行數
    row = cursor.rowcount 
        if row == 1: 
            # 提交資料庫的變更
            connection.commit()   
        else:
            connection.rolback() #取消該連線(上述)做得任何修改
            print('未改變資料')


    cursor.execute("SELECT * FROM user;")
    result = cursor.fetchall()

pprint(result)

刪除資料表

In [ ]:
import pymysql
from configparser import ConfigParser

config = ConfigParser()
config.read('../Chapter1/config.ini') # 指定設定檔的檔案路徑

connection = pymysql.connect(
    host=config.get('DB', 'host'),
    user=config.get('DB', 'user'),
    password=config.get('DB', 'password'),
    port=config.getint('DB', 'port'),
    cursorclass=pymysql.cursors.DictCursor,
    database = 'chapter2'
)

#如果資料跑很久無法刪除是因為上面的範例connection2 系統判定有資料還在連線中
#drop 不用commit
with connection.cursor() as cursor:
    sql = """
        DROP TABLLE user IF EXISTS user;
    """
    cursor.execute(sql)
    cursor.execute("SHOW TABLES;")
    result = cursor.fetchall()

print(result)

In [ ]:
# ipynb若沒關掉 變數會卡在資料庫裡(.py不會有這個問題)
connection.close()